# Solution 5 — Complete CSTR estimation, feedback and debugging

This worked solution covers every task: balances, nominal residual, nonlinear
versus linear simulation, mesh refinement, realistic offline initialization,
identifiability, held-out validation, temperature-only MHE, estimated-state NMPC,
soft temperature slack and deliberate failure diagnosis. All physical equations
and estimation/control transcriptions are local, so this solution runs from a
fresh kernel without importing a model or executing a chapter.


### Prepare an independent notebook

Run cells from top to bottom in a fresh kernel. This notebook defines its own
physical equations, estimation problem and numerical experiments; it does not
import a model or execute another notebook. Repository helpers only display
figures, create optional controls and summarize solver statistics.

NumPy arrays below store time along rows, while CasADi decision matrices store
states along rows and time along columns. `DM.full()` crosses the boundary from
CasADi numerical matrices to NumPy. Plotly provides hover, zoom and trace toggles;
`CASADI_TUTORIAL_PLOTS=static` selects Matplotlib and
`CASADI_TUTORIAL_WIDGETS=0` hides optional controls. All default results are
computed by ordinary functions before any Run button is created.


In [ ]:
import os
import sys
from pathlib import Path
from time import perf_counter
from functools import partial

import casadi as ca
import matplotlib.pyplot as plt
import numpy as np

repository_root = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from casadi_tutorial.diagnostics import max_bound_violation, solver_summary
from casadi_tutorial.interactive import experiment
from casadi_tutorial.plotting import display_plot, plot_series

# Every trajectory time axis in the CSTR lessons is measured in hours.
plot_series = partial(plot_series, xlabel="time [h]")

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

np.set_printoptions(precision=5, suppress=True)


### Derive the well-mixed reactor balances and fix the units

Assume constant liquid volume and density, equal inlet/outlet flow, perfect
mixing, one irreversible first-order reaction, and an imposed jacket temperature.
The state is $x=[C_A,T]^\mathsf T\in\mathbb R^2$, input $u=T_c\in\mathbb R$,
known disturbance $d=[C_{Af},T_f]^\mathsf T\in\mathbb R^2$, and the uncertain
parameter is total heat-transfer conductance $U_A$.

| Quantity | Value | Units |
|---|---:|---|
| Flow $F$, volume $V$ | 1, 1 | m³/h, m³ |
| Arrhenius prefactor $k_0$ | $9703\times3600$ | h⁻¹ |
| Activation energy $E$, gas constant $R$ | 11843, 1.98589 | kcal/kmol, kcal/(kmol K) |
| Reaction enthalpy $\Delta H$ | −5960 | kcal/kmol |
| Volumetric heat capacity $\rho C_p$ | 500 | kcal/(m³ K) |
| True $U_A$ | 150 | kcal/(h K) |
| Feed $C_{Af},T_f$ | 10, 298.15 | kmol/m³, K |

The rate and material/energy balances are

$$r=k_0e^{-E/(RT)}C_A,\qquad
\dot C_A=\frac FV(C_{Af}-C_A)-r,$$
$$\dot T=\frac FV(T_f-T)-\frac{\Delta H}{\rho C_p}r
+\frac{U_A}{V\rho C_p}(T_c-T).$$

Each material-balance term has units kmol/(m³ h); each energy-balance term has
units K/h. Because $\Delta H<0$, reaction releases heat. The jacket removes heat
when $T_c<T$. Concentration and absolute temperature must remain physical; a
numerically converged trajectory outside the stated domain is unsuitable.

**Predict:** increasing the jacket temperature initially changes which balance?
Which sign in the energy balance would accidentally turn an exothermic reaction
into cooling?


In [ ]:
FLOW = 1.0
VOLUME = 1.0
K0 = 9703.0 * 3600.0
ACTIVATION_ENERGY = 11843.0
GAS_CONSTANT = 1.98589
REACTION_ENTHALPY = -5960.0
HEAT_CAPACITY = 500.0
TRUE_UA = 150.0
X_NOMINAL = np.array([8.5695, 311.267])
U_NOMINAL = 298.15
D_NOMINAL = np.array([10.0, 298.15])
X_SCALE = np.array([0.5, 5.0])
U_SCALE = 10.0
STATE_LOWER = np.array([0.0, 285.0])
STATE_UPPER = np.array([12.0, 360.0])
INPUT_LOWER, INPUT_UPPER = 285.0, 315.0


def cstr_rhs(state, jacket_temperature, heat_transfer, disturbance):
    """Visible species/energy balances; time in h, temperature in K."""
    concentration, temperature = state[0], state[1]
    feed_concentration, feed_temperature = disturbance[0], disturbance[1]
    rate = K0 * ca.exp(-ACTIVATION_ENERGY / (GAS_CONSTANT * temperature)) * concentration
    material_balance = FLOW / VOLUME * (feed_concentration - concentration) - rate
    energy_balance = (
        FLOW / VOLUME * (feed_temperature - temperature)
        - REACTION_ENTHALPY / HEAT_CAPACITY * rate
        + heat_transfer / (VOLUME * HEAT_CAPACITY) * (jacket_temperature - temperature)
    )
    return ca.vertcat(material_balance, energy_balance)


### Turn the balances into differentiable continuous and discrete maps

Named CasADi inputs make the dimensions explicit: `x` is 2×1, `u` and `ua`
are scalars, `d` is 2×1, and `h` is a scalar measured in hours. Keeping `ua` as a
function input lets the same graph participate in parameter estimation.

A zero-order hold keeps $u,d$ fixed over each interval. Classical RK4 computes
$k_1=f(x)$, $k_2=f(x+hk_1/2)$, $k_3=f(x+hk_2/2)$,
$k_4=f(x+hk_3)$ and
$F_h=x+h(k_1+2k_2+2k_3+k_4)/6$. This approximation is differentiable but its
accuracy must be checked separately from solver convergence.

`jacobian` gives $A=\partial f/\partial x$, $B=\partial f/\partial u$ and
$E_d=\partial f/\partial d$. They describe perturbations about an operating
point, not absolute state evolution.


In [ ]:
x_symbol = ca.MX.sym("x", 2)
u_symbol = ca.MX.sym("u")
ua_symbol = ca.MX.sym("ua")
d_symbol = ca.MX.sym("d", 2)
h_symbol = ca.MX.sym("h")
xdot = cstr_rhs(x_symbol, u_symbol, ua_symbol, d_symbol)
f = ca.Function("cstr_balances", [x_symbol, u_symbol, ua_symbol, d_symbol], [xdot],
                ["x", "u", "ua", "d"], ["xdot"])
k1 = xdot
k2 = cstr_rhs(x_symbol + h_symbol * k1 / 2, u_symbol, ua_symbol, d_symbol)
k3 = cstr_rhs(x_symbol + h_symbol * k2 / 2, u_symbol, ua_symbol, d_symbol)
k4 = cstr_rhs(x_symbol + h_symbol * k3, u_symbol, ua_symbol, d_symbol)
F = ca.Function("cstr_rk4", [x_symbol, u_symbol, ua_symbol, d_symbol, h_symbol],
                [x_symbol + h_symbol * (k1 + 2*k2 + 2*k3 + k4) / 6],
                ["x", "u", "ua", "d", "h"], ["x_next"])
linearization = ca.Function("cstr_jacobians", [x_symbol, u_symbol, ua_symbol, d_symbol],
    [ca.jacobian(xdot, x_symbol), ca.jacobian(xdot, u_symbol), ca.jacobian(xdot, d_symbol)])


### Keep simulation, validation and presentation separate

`simulate_cstr` returns an $(N+1)\times2$ numerical history for $N$ held inputs.
The first row is the initial state, so measurements and state plots have one
more point than the input sequence. The held-input plot repeats its final value
at the last state time so the last interval remains visible.

`constraint_violation` independently evaluates every Opti constraint against its
bounds in the solver's scaled coordinates. A successful return status alone
does not check finite numbers or physical constraints. Objective normalization
is distinct from decision/constraint scaling; subsequent NLPs use dimensionless
state decisions $z=S_x^{-1}(x-x_s)$, $S_x=\operatorname{diag}(0.5,5)$.


In [ ]:
IPOPT_OPTIONS = {"print_level": 0, "sb": "yes", "max_iter": 200, "tol": 1e-7,
                 "bound_relax_factor": 0.0, "constr_viol_tol": 1e-8}


def simulate_cstr(initial_state, inputs, heat_transfer=TRUE_UA, step_size=0.1,
                  disturbance=D_NOMINAL):
    states = [np.asarray(initial_state, dtype=float).copy()]
    for jacket in np.asarray(inputs).reshape(-1):
        states.append(np.asarray(F(states[-1], jacket, heat_transfer, disturbance, step_size)).ravel())
    return np.asarray(states)


def constraint_violation(problem, solution):
    return max_bound_violation(
        np.asarray(solution.value(problem.g)).ravel(),
        np.asarray(solution.value(problem.lbg)).ravel(),
        np.asarray(solution.value(problem.ubg)).ravel(),
    )


def show_trajectory(states, inputs, step_size, title, measured_temperature=None):
    time = step_size * np.arange(len(states))
    display_plot(plot_series(time, {"concentration": states[:, 0]},
        title=title, ylabel="C_A [kmol/m³]"))
    temperatures = {"temperature": states[:, 1]}
    if measured_temperature is not None:
        temperatures["noisy observation"] = measured_temperature
    display_plot(plot_series(time, temperatures, title=title, ylabel="T [K]"))
    if len(inputs):
        held = np.r_[inputs, inputs[-1]]
        display_plot(plot_series(time, {"jacket": held},
            title=title, ylabel="T_c [K]", steps=True))


### Check the operating point before estimating or controlling

An equilibrium satisfies $f(x_s,u_s,d_s;U_A)=0$. The printed residual contains
kmol/(m³ h) and K/h; its small nonzero value reflects the rounded documented
state. Eigenvalues of $A$ have units h⁻¹ and determine local time scales.
A stable local linearization does not imply global stability: thermal feedback
can create other steady states.

**Interpret:** compare the slow time constant with the simulation duration.
Would five updates at 0.05 h reveal a settled response?


In [ ]:
nominal_residual = np.asarray(f(X_NOMINAL, U_NOMINAL, TRUE_UA, D_NOMINAL)).ravel()
A, B, E_disturbance = [np.asarray(item) for item in
    linearization(X_NOMINAL, U_NOMINAL, TRUE_UA, D_NOMINAL)]
eigenvalues = np.linalg.eigvals(A)
print("nominal residual [kmol/(m³ h), K/h]:", nominal_residual)
print("A [mixed state units/h]:\n", A)
print("eigenvalues [1/h]:", eigenvalues)
print("slow local time constant [h]:", -1 / np.max(eigenvalues.real))
assert np.linalg.norm(nominal_residual, np.inf) < 2e-4
assert A.shape == (2, 2) and B.shape == (2, 1) and E_disturbance.shape == (2, 2)
assert np.all(eigenvalues.real < 0)


### Compare nonlinear and linear step responses

The perturbation model is $\delta\dot x=A\delta x+B\delta u+E_d\delta d$.
Both simulations start at the nominal state and receive the same jacket step;
only the nonlinear trajectory changes its local derivatives as it moves.
The linear RK4 recurrence below evolves $\delta x$ and then adds $x_s$ back.

The discrepancy should grow for larger steps. A good fit near nominal operation
is evidence of local consistency, not permission to extrapolate. Use the
optional experiment to change the step amplitude after inspecting the default.


In [ ]:
def compare_linear_response(step_amplitude=5.0, step_size=0.05, steps=60, show=True):
    inputs = np.full(steps, U_NOMINAL + step_amplitude)
    nonlinear = simulate_cstr(X_NOMINAL, inputs, step_size=step_size)
    perturbation = np.zeros(2)
    linear = [X_NOMINAL.copy()]
    for _ in inputs:
        def perturbation_rhs(value):
            return A @ value + B[:, 0] * step_amplitude
        a = perturbation_rhs(perturbation)
        b = perturbation_rhs(perturbation + step_size * a / 2)
        c = perturbation_rhs(perturbation + step_size * b / 2)
        d = perturbation_rhs(perturbation + step_size * c)
        perturbation = perturbation + step_size * (a + 2*b + 2*c + d) / 6
        linear.append(X_NOMINAL + perturbation)
    linear = np.asarray(linear)
    if show:
        time = step_size * np.arange(steps + 1)
        for component, unit in [(0, "C_A [kmol/m³]"), (1, "T [K]")]:
            display_plot(plot_series(time, {"nonlinear": nonlinear[:, component],
                "linearized": linear[:, component]}, title="Local model comparison", ylabel=unit))
        display_plot(plot_series(time, {"jacket": np.r_[inputs, inputs[-1]]},
            title="Held experimental input", ylabel="T_c [K]", steps=True))
    return nonlinear, linear

nonlinear_response, linear_response = compare_linear_response()
print("maximum linear-model error [kmol/m³, K]:",
      np.max(np.abs(nonlinear_response - linear_response), axis=0))
assert np.isfinite(nonlinear_response).all() and np.isfinite(linear_response).all()
large_nonlinear, large_linear = compare_linear_response(step_amplitude=15.0, show=False)
large_step_error = np.max(np.abs(large_nonlinear - large_linear), axis=0)
print("maximum error for a +15 K step [kmol/m³, K]:", large_step_error)
assert np.all(large_step_error > np.max(np.abs(nonlinear_response - linear_response), axis=0))
def linear_response_experiment(step_amplitude=5.0):
    compare_linear_response(step_amplitude=step_amplitude)

if widgets is not None:
    experiment(linear_response_experiment, {
        "step_amplitude": widgets.FloatSlider(value=5, min=-10, max=15, step=1,
            description="jacket step [K]", continuous_update=False),
    }, title="Linear-model validity")


### Verify the integration mesh, independently of IPOPT

For the same physical 6 h experiment, halve the RK4 step while repeating each
held input. Compare only coincident sample times; comparing unrelated rows would
confuse time alignment with integration error. Agreement under refinement tests
this particular operating region, not every high-temperature scenario.

**Interpret:** if the optimization tolerance is $10^{-7}$ but the dynamics change
by $10^{-2}$ under refinement, which error should be addressed first?


In [ ]:
CALIBRATION_DT = 0.1
calibration_inputs = np.repeat(U_NOMINAL + np.array([0.0, 5.0, -5.0, 3.0]), 15)
mesh_trajectories = [simulate_cstr(X_NOMINAL, np.repeat(calibration_inputs, factor),
    step_size=CALIBRATION_DT / factor)[::factor] for factor in (1, 2, 4)]
mesh_errors = [np.max(np.abs(mesh_trajectories[index] - mesh_trajectories[-1]), axis=0)
               for index in (0, 1)]
for label, error in zip(("h versus h/4", "h/2 versus h/4"), mesh_errors):
    print(label, "errors [kmol/m³, K]:", [f"{value:.3e}" for value in error])
assert np.max(mesh_errors[0]) < 1e-3
assert np.max(mesh_errors[1]) < np.max(mesh_errors[0])


### Formulate an offline, bounded heat-transfer fit

For this calibration experiment the reactor is initially settled at the known
nominal state. Subsequent state estimation will deliberately lack this knowledge.
Observe only $y_k=T_k+\epsilon_k$, $\epsilon_k\sim\mathcal N(0,\sigma_T^2)$.
The decision variables are $U_A$ and $z\in\mathbb R^{2\times(N+1)}$, with
$x_k=x_s+S_xz_k$. Solve

$$\min_{U_A,z}\sum_{k=0}^N((T_k-y_k)/\sigma_T)^2$$
$$x_0=x_s,\quad S_x^{-1}(x_{k+1}-F_h(x_k,u_k,d_s;U_A))=0,
\quad50\le U_A\le300.$$

State bounds retain the model's operating region. Both state decisions and
continuity residuals are normalized; the parameter decision is normalized by
150 kcal/(h K). The initial trajectory is simulated with a guessed parameter,
not copied from the synthetic truth. The fitted trajectory is an offline smoother
using all measurements, not a causal state estimator.


In [ ]:
def fit_heat_transfer(inputs, measured_temperature, noise_sigma=0.05, initial_guess=100.0,
                      step_size=0.1):
    inputs = np.asarray(inputs)
    optimization = ca.Opti()
    scaled_heat_transfer = optimization.variable()
    heat_transfer = 150.0 * scaled_heat_transfer
    z = optimization.variable(2, len(inputs) + 1)
    states = ca.repmat(ca.DM(X_NOMINAL), 1, len(inputs) + 1) + ca.diag(ca.DM(X_SCALE)) @ z
    optimization.subject_to(z[:, 0] == 0)
    optimization.subject_to(optimization.bounded(50, heat_transfer, 300))
    optimization.subject_to(optimization.bounded(0, states[0, :], 12))
    optimization.subject_to(optimization.bounded(285, states[1, :], 360))
    for index, jacket in enumerate(inputs):
        defect = states[:, index + 1] - F(states[:, index], jacket, heat_transfer,
                                          D_NOMINAL, step_size)
        optimization.subject_to(ca.diag(ca.DM(1 / X_SCALE)) @ defect == 0)
    residual = (states[1, :] - ca.DM(measured_temperature).T) / noise_sigma
    optimization.minimize(ca.sumsqr(residual))
    guessed_states = simulate_cstr(X_NOMINAL, inputs, initial_guess, step_size)
    optimization.set_initial(z, ((guessed_states - X_NOMINAL) / X_SCALE).T)
    optimization.set_initial(scaled_heat_transfer, initial_guess / 150)
    optimization.solver("ipopt", {"print_time": False}, IPOPT_OPTIONS)
    solution = optimization.solve()
    result = {"ua": float(solution.value(heat_transfer)),
              "states": np.asarray(solution.value(states)).T,
              "objective": float(solution.value(optimization.f)),
              "primal_inf": constraint_violation(optimization, solution),
              **solver_summary(solution.stats())}
    if not result["success"] or not np.isfinite(result["states"]).all() or result["primal_inf"] > 1e-5:
        raise RuntimeError("Heat-transfer fit failed independent validation")
    return result


### Fit six hours of temperature data without an oracle initial guess

Four 1.5 h jacket plateaus generate a reproducible experiment. The random seed
fixes measurement noise so changes to the model or initialization can be compared.
Truth is used to create synthetic observations and evaluate the result; it is not
passed into the optimizer. The fit starts at $U_A=100$, while truth is 150.

**Interpret:** why can a fit initialized from the exact true trajectory hide a
poor initialization strategy? Why does the residual denominator have units K?


In [ ]:
CALIBRATION_DT = 0.1
calibration_inputs = np.repeat(U_NOMINAL + np.array([0.0, 5.0, -5.0, 3.0]), 15)
calibration_truth = simulate_cstr(X_NOMINAL, calibration_inputs, TRUE_UA, CALIBRATION_DT)
measurement_sigma = 0.05
calibration_measurements = calibration_truth[:, 1] + np.random.default_rng(7).normal(
    0, measurement_sigma, len(calibration_truth))
calibration = fit_heat_transfer(calibration_inputs, calibration_measurements,
                               measurement_sigma, initial_guess=100, step_size=CALIBRATION_DT)
FITTED_UA = calibration["ua"]
print("heat-transfer fit:", {key: value for key, value in calibration.items() if key != "states"})
assert abs(FITTED_UA - TRUE_UA) / TRUE_UA < 0.05
assert calibration["primal_inf"] < 1e-6


### Inspect residuals and validate on a new experiment

A training fit can absorb noise or compensate for an incorrect model. A separate
3 h jacket profile tests prediction using the fitted parameter without refitting.
The held-out seed and measurements differ from the calibration data. Temperature
prediction RMSE is reported in K, separately from normalized least-squares cost.
Residual patterns matter: drift or input-correlated errors suggest missing physics.


In [ ]:
calibration_time = CALIBRATION_DT * np.arange(len(calibration_measurements))
for component, unit in [(0, "C_A [kmol/m³]"), (1, "T [K]")]:
    series = {"synthetic truth": calibration_truth[:, component],
              "offline fit": calibration["states"][:, component]}
    if component == 1:
        series["observed temperature"] = calibration_measurements
    display_plot(plot_series(calibration_time, series, title="Offline fitted states", ylabel=unit))
display_plot(plot_series(calibration_time, {"jacket": np.r_[calibration_inputs, calibration_inputs[-1]]},
    title="Calibration input", ylabel="T_c [K]", steps=True))
calibration_time = CALIBRATION_DT * np.arange(len(calibration_measurements))
display_plot(plot_series(calibration_time,
    {"observed minus fitted": calibration_measurements - calibration["states"][:, 1]},
    title="Calibration residuals", ylabel="temperature residual [K]"))
heldout_inputs = np.repeat(U_NOMINAL + np.array([2.0, -4.0, 6.0]), 10)
heldout_truth = simulate_cstr(X_NOMINAL, heldout_inputs, TRUE_UA, CALIBRATION_DT)
heldout_measurements = heldout_truth[:, 1] + np.random.default_rng(19).normal(
    0, measurement_sigma, len(heldout_truth))
heldout_prediction = simulate_cstr(X_NOMINAL, heldout_inputs, FITTED_UA, CALIBRATION_DT)
heldout_rmse = float(np.sqrt(np.mean((heldout_prediction[:, 1] - heldout_measurements)**2)))
print("held-out temperature prediction RMSE [K]:", heldout_rmse)
assert heldout_rmse < 0.1
show_trajectory(heldout_prediction, heldout_inputs, CALIBRATION_DT,
                "Held-out prediction: no refitting", heldout_measurements)


### Separate convergence from information about the parameter

Fix a candidate $U_A$, simulate its trajectory, and profile the same normalized
measurement objective. The curve shows whether data distinguish nearby values;
a solver status cannot answer that question. Finite differences of the complete
simulation provide temperature sensitivity $\partial T_k/\partial U_A$.

With all other physics and the initial state known, even constant nominal jacket
input contains information about $U_A$: heat exchange is nonzero. Do not infer
unidentifiability from the absence of input changes alone. Shorter/noisier records
reduce information; adding uncertain kinetic parameters can introduce strong
correlations. Any local uncertainty approximation also assumes the model and
noise assumptions are correct.


In [ ]:
profile_grid = np.linspace(120, 180, 41)
profile_objective = np.array([np.sum(((simulate_cstr(X_NOMINAL, calibration_inputs, value,
    CALIBRATION_DT)[:, 1] - calibration_measurements) / measurement_sigma)**2)
    for value in profile_grid])
display_plot(plot_series(profile_grid, {"profile objective": profile_objective},
    title="Heat-transfer objective profile", xlabel="U_A [kcal/(h K)]",
    ylabel="normalized squared residual"))
parameter_step = 0.1
sensitivity = (simulate_cstr(X_NOMINAL, calibration_inputs, FITTED_UA + parameter_step,
    CALIBRATION_DT)[:, 1] - simulate_cstr(X_NOMINAL, calibration_inputs,
    FITTED_UA - parameter_step, CALIBRATION_DT)[:, 1]) / (2 * parameter_step)
print("temperature sensitivity norm [K/(kcal/(h K))]:", np.linalg.norm(sensitivity))
print("local one-parameter noise-only standard error [kcal/(h K)]:",
      measurement_sigma / np.linalg.norm(sensitivity))
display_plot(plot_series(calibration_time, {"temperature sensitivity": sensitivity},
    title="Sensitivity of predicted temperature to fitted heat transfer",
    ylabel="dT/dU_A [K/(kcal/(h K))]"))
assert np.isfinite(profile_objective).all() and np.linalg.norm(sensitivity) > 0


### Explain why an unmeasured concentration can be reconstructed

For temperature measurements $H=[0,1]$, local observability requires
$\operatorname{rank}([H;HA])=2$. The concentration changes reaction heat, which
then changes measured temperature. The rank test is local; small sensitivities
and noisy data can still make practical reconstruction poor. Uncertain kinetics
or heat transfer may be confounded with concentration, motivating offline
calibration before online estimation.

**Interpret:** what would happen to concentration information if reaction heat
were negligible? Why is differentiating noisy temperature a fragile alternative?


In [ ]:
A_observe = np.asarray(linearization(X_NOMINAL, U_NOMINAL, FITTED_UA, D_NOMINAL)[0])
H = np.array([[0.0, 1.0]])
observability = np.vstack([H, H @ A_observe])
print("temperature-only local observability matrix:\n", observability)
print("rank:", np.linalg.matrix_rank(observability))
assert np.linalg.matrix_rank(observability) == 2


### Formulate temperature-only moving-horizon state estimation

Only reactor temperature is measured online. Inputs already applied to the plant
are known. Hold the offline fitted $\widehat U_A$ fixed and estimate both
concentration and temperature over the most recent $m\le10$ intervals:

$$\min_{x,w}\|D_a^{-1}(x_0-\bar x_0)\|^2+
\sum_{j=0}^{m-1}\|D_w^{-1}w_j\|^2+
\sum_{j=0}^{m}((T_j-y_j)/\sigma_T)^2$$
$$x_{j+1}=F_h(x_j,u_j,d_s;\widehat U_A)+w_j,\quad
0\le C_{A,j}\le12,\quad285\le T_j\le360.$$

The arrival scales are $D_a=\operatorname{diag}(1\;\mathrm{kmol/m^3},0.5\;\mathrm K)$;
model-error scales per interval are
$D_w=\operatorname{diag}(0.002\;\mathrm{kmol/m^3},0.015\;\mathrm K)$;
$\sigma_T=0.05$ K by default. Smaller scales penalize the corresponding residual
more strongly. These are engineering weights: the plant need not contain white
process noise, and fixed arrival weights do not constitute an exact Bayesian
posterior or calibrated uncertainty bands.

The $2(m+1)$ scaled state decisions and $2m$ scaled model-error decisions keep
heterogeneous units out of the numerical coordinates. The objective includes
$m+1$ measurements but only $m$ applied inputs; the newest measurement precedes
the next control decision.


In [ ]:
ARRIVAL_SCALE = np.array([1.0, 0.5])
MODEL_ERROR_SCALE = np.array([0.002, 0.015])


def build_mhe_problem(intervals, model_ua, step_size, noise_sigma):
    optimization = ca.Opti()
    z = optimization.variable(2, intervals + 1)
    states = ca.repmat(ca.DM(X_NOMINAL), 1, intervals + 1) + ca.diag(ca.DM(X_SCALE)) @ z
    scaled_error = optimization.variable(2, intervals) if intervals else None
    observations = optimization.parameter(1, intervals + 1)
    applied = optimization.parameter(1, intervals) if intervals else None
    arrival_mean = optimization.parameter(2)
    arrival_residual = ca.diag(ca.DM(1 / ARRIVAL_SCALE)) @ (states[:, 0] - arrival_mean)
    cost = ca.sumsqr(arrival_residual) + ca.sumsqr((states[1, :] - observations) / noise_sigma)
    for index in range(intervals):
        model_error = ca.diag(ca.DM(MODEL_ERROR_SCALE)) @ scaled_error[:, index]
        defect = states[:, index + 1] - F(states[:, index], applied[:, index],
            model_ua, D_NOMINAL, step_size) - model_error
        optimization.subject_to(ca.diag(ca.DM(1 / X_SCALE)) @ defect == 0)
    if intervals:
        cost += ca.sumsqr(scaled_error)
    optimization.subject_to(optimization.bounded(0, states[0, :], 12))
    optimization.subject_to(optimization.bounded(285, states[1, :], 360))
    optimization.minimize(cost)
    optimization.solver("ipopt", {"print_time": False}, IPOPT_OPTIONS)
    return {"optimization": optimization, "z": z, "states": states,
            "scaled_error": scaled_error, "observations": observations,
            "applied": applied, "arrival_mean": arrival_mean}


### Grow the startup window and update the estimator causally

There is no invented measurement prehistory. At sample $k$, use
$m=\min(k,10)$ and cache the eleven possible problem sizes. Before the window is
full, the arrival mean is the declared initial prior. Once the window advances,
use the previous estimate at its new oldest node. This fixed-weight shifted
arrival cost is an approximation and may reuse information; compare its weights
rather than interpreting it as an exact posterior.

Shift/extend the previous estimated trajectory for initialization. `mhe_step`
receives only measurements, past inputs and estimator memory: it cannot access
simulated true states. Reject failed/nonfinite/infeasible results. One failure
propagates the previous estimate using the applied input and marks the update as
degraded; a second consecutive failure stops the experiment. The fallback is an
educational prediction, not a safety guarantee.


In [ ]:
def new_mhe(model_ua, step_size=0.1, noise_sigma=0.05, window=10,
            initial_prior=None):
    return {"model_ua": model_ua, "step_size": step_size, "noise_sigma": noise_sigma,
            "window": window, "prior": np.array([8.0, 309.0]) if initial_prior is None
            else np.asarray(initial_prior, dtype=float), "problems": {},
            "previous_states": None, "consecutive_failures": 0}


def mhe_step(estimator, observations, applied_inputs, force_failure=False):
    sample = len(observations) - 1
    intervals = min(sample, estimator["window"])
    if len(applied_inputs) != sample:
        raise ValueError("One fewer applied input than temperature observations is required")
    if intervals not in estimator["problems"]:
        estimator["problems"][intervals] = build_mhe_problem(intervals,
            estimator["model_ua"], estimator["step_size"], estimator["noise_sigma"])
    problem = estimator["problems"][intervals]
    optimization = problem["optimization"]
    previous = estimator["previous_states"]
    if previous is None:
        guess = estimator["prior"][None, :]
    else:
        next_guess = np.asarray(F(previous[-1], applied_inputs[-1], estimator["model_ua"],
                                 D_NOMINAL, estimator["step_size"])).ravel()
        guess = np.vstack([previous, next_guess])[-intervals-1:]
    arrival = estimator["prior"] if sample <= estimator["window"] else previous[1]
    optimization.set_value(problem["arrival_mean"], arrival)
    optimization.set_value(problem["observations"], np.asarray(observations[-intervals-1:])[None, :])
    if intervals:
        optimization.set_value(problem["applied"], np.asarray(applied_inputs[-intervals:])[None, :])
        optimization.set_initial(problem["scaled_error"], 0)
    optimization.set_initial(problem["z"], ((guess - X_NOMINAL) / X_SCALE).T)
    try:
        if force_failure:
            raise RuntimeError("Injected unavailable estimator result")
        solution = optimization.solve()
        states = np.asarray(solution.value(problem["states"])).reshape(2, intervals + 1).T
        diagnostics = {**solver_summary(solution.stats()),
                       "primal_inf": constraint_violation(optimization, solution), "degraded": False}
        if not diagnostics["success"] or not np.isfinite(states).all() or diagnostics["primal_inf"] > 1e-5:
            raise RuntimeError("Estimator result failed independent validation")
        estimator["consecutive_failures"] = 0
    except RuntimeError as error:
        estimator["consecutive_failures"] += 1
        if estimator["consecutive_failures"] > 1:
            raise RuntimeError("Stopping after two consecutive estimator failures") from error
        states = guess
        if not np.isfinite(states).all() or np.any(states[-1] < STATE_LOWER) or np.any(states[-1] > STATE_UPPER):
            raise RuntimeError("Propagated estimator fallback is outside the model domain") from error
        diagnostics = {"success": False, "return_status": str(error).splitlines()[-1],
                       "iterations": None, "primal_inf": np.nan, "degraded": True}
    estimator["previous_states"] = states
    return states[-1].copy(), diagnostics


### Estimate a trajectory from noisy temperature and known applied inputs

The plant starts at $x_s+[0.5,-2]$, while the estimator starts from $[8,309]$;
its concentration error is deliberately substantial. The plant simulator creates
a record once. The estimator loop subsequently receives only that record's noisy
temperatures and input history. No true state is passed to `mhe_step`.

Report state RMSE after the 1 h startup separately from all-time error. The
initial prior error is part of the task, not evidence that the numerical solver
failed. Synthetic truth is used exclusively for these evaluation plots/metrics.


In [ ]:
def run_state_estimation(noise=0.05, window=10, model_ua=None, failure_steps=()):
    model_ua = FITTED_UA if model_ua is None else model_ua
    inputs = np.repeat(U_NOMINAL + np.array([0.0, 5.0, -5.0, 3.0]), 15)
    truth = simulate_cstr(X_NOMINAL + np.array([0.5, -2.0]), inputs, TRUE_UA, 0.1)
    observations = truth[:, 1] + np.random.default_rng(42).normal(0, noise, len(truth))
    estimator = new_mhe(model_ua, noise_sigma=noise, window=window)
    estimates, diagnostics = [], []
    for sample in range(len(observations)):
        estimate, report = mhe_step(estimator, observations[:sample+1], inputs[:sample],
                                    force_failure=sample in failure_steps)
        estimates.append(estimate)
        diagnostics.append(report)
    estimates = np.asarray(estimates)
    rmse = np.sqrt(np.mean((estimates[10:] - truth[10:])**2, axis=0))
    return {"truth": truth, "estimates": estimates, "observations": observations,
            "inputs": inputs, "diagnostics": diagnostics, "rmse": rmse, "step_size": 0.1}

state_estimation = run_state_estimation()
print("MHE RMSE after 1 h [kmol/m³, K]:", state_estimation["rmse"])
assert np.all(state_estimation["rmse"] < np.array([0.08, 0.06]))
assert all(report["success"] for report in state_estimation["diagnostics"])


### Examine errors, observations and solver health together

Plots distinguish true states, causal MHE estimates, noisy temperature and the
input that actually generated the observations. The estimation-error plot must
shrink without giving the estimator access to true concentration. Iteration
counts reveal computational changes that a smooth state plot can hide.


In [ ]:
def show_state_estimation(result):
    time = result["step_size"] * np.arange(len(result["truth"]))
    for component, unit in [(0, "C_A [kmol/m³]"), (1, "T [K]")]:
        series = {"true": result["truth"][:, component], "MHE": result["estimates"][:, component]}
        if component == 1:
            series["temperature observation"] = result["observations"]
        display_plot(plot_series(time, series, title="Temperature-only state estimation", ylabel=unit))
        display_plot(plot_series(time,
            {"estimate minus truth": result["estimates"][:, component] - result["truth"][:, component]},
            title="Causal estimation error", ylabel=unit))
    display_plot(plot_series(time, {"jacket": np.r_[result["inputs"], result["inputs"][-1]]},
        title="Known applied input", ylabel="T_c [K]", steps=True))
    iterations = [np.nan if item["iterations"] is None else item["iterations"]
                  for item in result["diagnostics"]]
    display_plot(plot_series(time, {"MHE iterations": iterations}, title="Estimator solve health",
                            ylabel="iterations"))

show_state_estimation(state_estimation)


### Exercise estimator degradation and a deliberate stop

This lab injects an unavailable result, rather than pretending an arbitrary
exception identifies a physical fault. One missing update uses model propagation
and is labeled degraded; the following successful update returns to measurement
feedback. Two consecutive failures terminate the experiment instead of silently
using true states. The controller chapter also triggers an actual iteration-limit
failure and inspects the solver's return status.

**Diagnose:** compare the error during the degraded step with ordinary MHE error.
Why would indefinitely propagating a model be unreliable under parameter mismatch?


In [ ]:
one_missing_update = run_state_estimation(failure_steps=(12,))
assert one_missing_update["diagnostics"][12]["degraded"]
assert one_missing_update["diagnostics"][13]["success"]
print("Injected missing update:", one_missing_update["diagnostics"][12])
try:
    run_state_estimation(failure_steps=(12, 13))
except RuntimeError as error:
    print("Expected estimator stop:", error)
    assert "two consecutive" in str(error)
else:
    raise AssertionError("Consecutive missing estimates must stop this experiment")


### Formulate a scaled, constrained receding-horizon controller

Let $z_k=S_x^{-1}(x_k-x_s)$ and $v_k=(u_k-u_s)/10$. At every sample solve

$$\min_{z,v}\sum_{j=0}^{N}\|z_j\|^2+0.1\sum_{j=0}^{N-1}v_j^2,$$
$$x_0=\widehat x_k,\quad S_x^{-1}(x_{j+1}-F_h(x_j,u_j,d_s;\widehat U_A))=0,$$
$$0\le C_{A,j}\le12,\quad285\le T_j\le360,\quad285\le u_j\le315.$$

The state estimate is a 2×1 Opti parameter; changing it does not rebuild the
$2\times(N+1)$ state and $1\times N$ input decisions. The terminal state receives
one tracking term and no hard terminal equality. With $h=0.1$ h and $N=20$, the
2 h prediction horizon is comparable to the local slow time constant. Sixty
updates give a 6 h recovery demonstration.

For the exercise, an optional softer operating limit adds nonnegative slack
$s_j$ in K: $T_j\le T_{\rm operating}+s_j$ and
$100\sum_j(s_j/1\mathrm K)^2$ to the cost. The 360 K hard model-domain bound
remains in force; a soft operating preference must not silently replace it.


In [ ]:
def build_controller(model_ua, step_size=0.1, horizon=20, max_iter=200,
                     soft_temperature_limit=None):
    optimization = ca.Opti()
    z = optimization.variable(2, horizon + 1)
    v = optimization.variable(1, horizon)
    states = ca.repmat(ca.DM(X_NOMINAL), 1, horizon + 1) + ca.diag(ca.DM(X_SCALE)) @ z
    inputs = U_NOMINAL + U_SCALE * v
    initial_state = optimization.parameter(2)
    optimization.subject_to(z[:, 0] == (initial_state - X_NOMINAL) / ca.DM(X_SCALE))
    for index in range(horizon):
        defect = states[:, index + 1] - F(states[:, index], inputs[:, index],
                                          model_ua, D_NOMINAL, step_size)
        optimization.subject_to(ca.diag(ca.DM(1 / X_SCALE)) @ defect == 0)
    optimization.subject_to(optimization.bounded(0, states[0, :], 12))
    optimization.subject_to(optimization.bounded(285, states[1, :], 360))
    optimization.subject_to(optimization.bounded(INPUT_LOWER, inputs, INPUT_UPPER))
    objective = ca.sumsqr(z) + 0.1 * ca.sumsqr(v)
    slack = None
    if soft_temperature_limit is not None:
        slack = optimization.variable(1, horizon + 1)
        optimization.subject_to(slack >= 0)
        optimization.subject_to(states[1, :] <= soft_temperature_limit + slack)
        objective += 100 * ca.sumsqr(slack / 1.0)
    optimization.minimize(objective)
    optimization.solver("ipopt", {"print_time": False}, {**IPOPT_OPTIONS, "max_iter": max_iter})
    return {"optimization": optimization, "z": z, "v": v, "states": states, "inputs": inputs,
            "initial_state": initial_state, "slack": slack, "horizon": horizon,
            "model_ua": model_ua, "step_size": step_size, "previous_prediction": None,
            "consecutive_failures": 0}


### Validate a control move and make failure behavior explicit

Warm-start with the previous state/input predictions shifted by one interval;
this suggests an initial guess without changing the next optimum. Apply only the
first optimized input. Record status, iterations, feasibility, solve time and
any slack before advancing the plant. Independently recompute physical dynamic
defects in kmol/m³ and K, outside the Opti constraint vector, and require their
maximum absolute component to be below $10^{-6}$ in these stated units.

If a solve fails, reject its decision values. A previously accepted next planned
input (or nominal input before the first accepted solve) is only a fallback
candidate. Check its bounds and one-step prediction from the current estimate;
stop if the candidate or current estimate fails domain checks. One degraded
move is permitted; two consecutive failed control solves stop the simulation.
These checks concern a teaching model and do not certify actual plant safety.


In [ ]:
def controller_move(controller, estimated_state):
    optimization = controller["optimization"]
    horizon = controller["horizon"]
    estimated_state = np.asarray(estimated_state, dtype=float)
    if (not np.isfinite(estimated_state).all() or np.any(estimated_state < STATE_LOWER)
            or np.any(estimated_state > STATE_UPPER)):
        raise RuntimeError("Initial estimate violates hard node-zero state bounds")
    previous = controller["previous_prediction"]
    if previous is None:
        input_guess = np.full(horizon, U_NOMINAL)
        state_guess = simulate_cstr(estimated_state, input_guess, controller["model_ua"],
                                   controller["step_size"])
    else:
        state_guess = np.vstack([previous["states"][1:], previous["states"][-1]])
        input_guess = np.r_[previous["inputs"][1:], previous["inputs"][-1]]
    optimization.set_value(controller["initial_state"], estimated_state)
    optimization.set_initial(controller["z"], ((state_guess - X_NOMINAL) / X_SCALE).T)
    optimization.set_initial(controller["v"], ((input_guess - U_NOMINAL) / U_SCALE)[None, :])
    if controller["slack"] is not None:
        optimization.set_initial(controller["slack"], 1.0)
    started = perf_counter()
    try:
        solution = optimization.solve()
        prediction = {"states": np.asarray(solution.value(controller["states"])).T,
                      "inputs": np.asarray(solution.value(controller["inputs"])).ravel()}
        # Evaluate physical defects independently of the scaled Opti constraint vector.
        physical_defects = np.array([
            prediction["states"][index + 1] - np.asarray(F(
                prediction["states"][index], prediction["inputs"][index],
                controller["model_ua"], D_NOMINAL, controller["step_size"])).ravel()
            for index in range(horizon)
        ])
        diagnostics = {**solver_summary(solution.stats()),
                       "primal_inf": constraint_violation(optimization, solution),
                       "dynamic_defect_inf": float(np.max(np.abs(physical_defects))),
                       "elapsed_s": perf_counter() - started, "degraded": False,
                       "max_slack": 0.0 if controller["slack"] is None
                       else float(np.max(solution.value(controller["slack"])))}
        if (not diagnostics["success"] or not np.isfinite(prediction["states"]).all()
                or not np.isfinite(prediction["inputs"]).all() or diagnostics["primal_inf"] > 1e-5
                or diagnostics["dynamic_defect_inf"] > 1e-6):
            raise RuntimeError("Controller solution failed independent validation")
        controller["previous_prediction"] = prediction
        controller["consecutive_failures"] = 0
        return float(prediction["inputs"][0]), diagnostics, prediction
    except RuntimeError as error:
        controller["consecutive_failures"] += 1
        if controller["consecutive_failures"] > 1:
            raise RuntimeError("Stopping after two consecutive controller failures") from error
        candidate = U_NOMINAL if previous is None else float(previous["inputs"][min(1, horizon-1)])
        next_estimate = np.asarray(F(estimated_state, candidate, controller["model_ua"],
                                     D_NOMINAL, controller["step_size"])).ravel()
        if (not INPUT_LOWER <= candidate <= INPUT_UPPER or not np.isfinite(next_estimate).all()
                or np.any(next_estimate < STATE_LOWER) or np.any(next_estimate > STATE_UPPER)):
            raise RuntimeError("No valid one-step fallback candidate; stopping") from error
        diagnostics = {**solver_summary(optimization.stats()), "primal_inf": np.nan,
            "dynamic_defect_inf": np.nan, "elapsed_s": perf_counter() - started, "degraded": True, "max_slack": np.nan,
            "message": str(error).splitlines()[-1]}
        return candidate, diagnostics, None


### Compare estimated-state feedback with ideal full-state feedback

Each branch gets its own plant simulation, the same fitted model, tuning and
initial actual state. The full-state branch knows $x_k$ exactly. The other branch
receives only $y_k=T_k+\epsilon_k$, estimates $(C_A,T)$, then supplies that estimate
to the same controller. Both apply only the first computed input before taking
the next measurement. True states are used solely inside each simulator and for
evaluation; `mhe_step` and `controller_move` never receive hidden true states.

Compare the integrated normalized tracking error
$h\sum_k\|S_x^{-1}(x_k-x_s)\|^2$, post-startup estimation RMSE, state/input
bounds and every solve status. Full-state feedback is a useful ideal baseline,
not the data available to the estimated-state controller.


In [ ]:
def run_control_comparison(noise=0.05, horizon=20, model_offset=0.0, moves=60):
    model_ua = FITTED_UA + model_offset
    result = {}
    for branch in ("full state", "estimated state"):
        plant = X_NOMINAL + np.array([0.5, -2.0])
        controller = build_controller(model_ua, horizon=int(horizon))
        estimator = new_mhe(model_ua, noise_sigma=noise)
        random_generator = np.random.default_rng(42)
        truth, estimates, inputs, observations = [plant.copy()], [], [], []
        control_reports, estimator_reports, predictions = [], [], []
        for sample in range(moves + 1):
            observations.append(float(plant[1] + random_generator.normal(0, noise)))
            if branch == "estimated state":
                estimate, report = mhe_step(estimator, observations, inputs)
                estimator_reports.append(report)
            else:
                estimate = plant.copy()
            estimates.append(estimate)
            if sample == moves:
                break
            applied, report, prediction = controller_move(controller, estimate)
            inputs.append(applied)
            control_reports.append(report)
            predictions.append(prediction)
            plant = np.asarray(F(plant, applied, TRUE_UA, D_NOMINAL, 0.1)).ravel()
            truth.append(plant.copy())
        truth, estimates = np.asarray(truth), np.asarray(estimates)
        result[branch] = {"truth": truth, "estimates": estimates, "inputs": np.asarray(inputs),
            "observations": np.asarray(observations), "control_reports": control_reports,
            "estimator_reports": estimator_reports, "predictions": predictions, "step_size": 0.1,
            "tracking": float(0.1 * np.sum(((truth - X_NOMINAL) / X_SCALE)**2)),
            "rmse": np.sqrt(np.mean((estimates[10:] - truth[10:])**2, axis=0))}
    return result

started = perf_counter()
comparison = run_control_comparison()
print("comparison elapsed [s]:", perf_counter() - started)
for branch, result in comparison.items():
    print(branch, "tracking:", result["tracking"], "post-startup RMSE:", result["rmse"],
          "final physical error:", result["truth"][-1] - X_NOMINAL)
    assert np.isfinite(result["truth"]).all()
    assert np.all(result["truth"] >= STATE_LOWER - 1e-6)
    assert np.all(result["truth"] <= STATE_UPPER + 1e-6)
    assert np.all((result["inputs"] >= INPUT_LOWER - 1e-6) & (result["inputs"] <= INPUT_UPPER + 1e-6))
    assert all(item["success"] and item["primal_inf"] < 1e-6
               and item["dynamic_defect_inf"] < 1e-6 for item in result["control_reports"])
assert np.all(comparison["estimated state"]["rmse"] < np.array([0.08, 0.06]))
assert comparison["estimated state"]["tracking"] < 1.2 * comparison["full state"]["tracking"]
assert all(item["success"] for item in comparison["estimated state"]["estimator_reports"])


### Inspect closed-loop performance and per-solve diagnostics

Overlay actual plant responses for both feedback policies and the states inferred
by MHE. Observe concentration reconstruction during startup, sensor noise after
startup and the timing of held inputs. Iterations and independent constraint
residuals expose solver behavior even when trajectories look similar.


In [ ]:
def show_control_comparison(results):
    ideal, estimated = results["full state"], results["estimated state"]
    time = estimated["step_size"] * np.arange(len(estimated["truth"]))
    for component, unit in [(0, "C_A [kmol/m³]"), (1, "T [K]")]:
        display_plot(plot_series(time, {"ideal feedback: true": ideal["truth"][:, component],
            "estimated feedback: true": estimated["truth"][:, component],
            "MHE state": estimated["estimates"][:, component],
            "target": np.full(len(time), X_NOMINAL[component])},
            title="CSTR feedback comparison", ylabel=unit))
    display_plot(plot_series(time, {
        "ideal feedback": np.r_[ideal["inputs"], ideal["inputs"][-1]],
        "estimated feedback": np.r_[estimated["inputs"], estimated["inputs"][-1]]},
        title="Actually applied jacket temperature", ylabel="T_c [K]", steps=True))
    for key, unit in [("iterations", "iterations"), ("primal_inf", "scaled violation"),
                      ("dynamic_defect_inf", "physical dynamic defect [state units]")]:
        display_plot(plot_series(time[:-1], {
            branch: [item[key] for item in result["control_reports"]]
            for branch, result in results.items()}, title="Controller solve diagnostics", ylabel=unit))

show_control_comparison(comparison)


### Trigger actual solver failures and check bounded recovery

An iteration budget of zero causes a genuine IPOPT failure. Inspect its status;
do not treat an iteration-limit failure as proof of infeasibility. The first
failure permits a checked nominal-input candidate; the second stops. A separate
node-zero domain test catches an inconsistent measured/estimated initial state
before a solver call. Increasing tolerance cannot reconcile a hard initial-state
contradiction.

**Diagnose:** use the reported status to decide whether to improve initialization,
allow more iterations, repair bounds or revisit the model. Chapter 18 adds
NaN/domain, derivative and scaling experiments.


In [ ]:
limited_controller = build_controller(FITTED_UA, max_iter=0)
fallback_input, failure_report, rejected_prediction = controller_move(
    limited_controller, X_NOMINAL + np.array([0.25, -1.0]))
print("Iteration-limit diagnostic:", failure_report)
print("Checked one-step fallback input [K]:", fallback_input)
assert failure_report["degraded"] and not failure_report["success"]
assert rejected_prediction is None and INPUT_LOWER <= fallback_input <= INPUT_UPPER
try:
    controller_move(limited_controller, X_NOMINAL + np.array([0.25, -1.0]))
except RuntimeError as error:
    print("Expected repeated-failure stop:", error)
    assert "two consecutive" in str(error)
else:
    raise AssertionError("Repeated failed controller solves must stop")
try:
    controller_move(build_controller(FITTED_UA), np.array([8.5, 370.0]))
except RuntimeError as error:
    print("Expected node-zero diagnosis:", error)
    assert "node-zero" in str(error)
else:
    raise AssertionError("An initial state outside hard bounds must be rejected")


### Complete the soft-temperature exercise and interpret slack

Impose a deliberately restrictive 308 K operating preference while retaining the
285–360 K hard physical range. With the actual initial temperature above 308 K,
the operating preference cannot hold at node zero. Nonnegative slack represents
that conflict in kelvin and pays a stated quadratic penalty; it does not change
the initial condition or make the violation disappear.

A good answer reports both physical temperature and slack, verifies the original
softened inequality and explains the remaining violation. A larger penalty does
not create more cooling authority or alter the reactor's instantaneous state.


In [ ]:
soft_controller = build_controller(FITTED_UA, soft_temperature_limit=308.0)
soft_input, soft_report, soft_prediction = controller_move(
    soft_controller, X_NOMINAL + np.array([0.5, -2.0]))
soft_solution = soft_controller["optimization"]
soft_slack = np.asarray(soft_solution.value(soft_controller["slack"])).ravel()
print("Soft-limit solve:", soft_report)
print("Maximum physical temperature [K]:", np.max(soft_prediction["states"][:, 1]))
print("Maximum slack [K]:", np.max(soft_slack))
assert soft_report["success"] and soft_report["max_slack"] > 0.1
assert soft_report["dynamic_defect_inf"] < 1e-6
assert np.max(soft_prediction["states"][:, 1] - 308.0 - soft_slack) < 1e-5
assert np.all(soft_prediction["states"][:, 1] <= 360.0 + 1e-6)
soft_time = 0.1 * np.arange(len(soft_slack))
display_plot(plot_series(soft_time, {"predicted temperature": soft_prediction["states"][:, 1],
    "operating limit plus slack": 308.0 + soft_slack}, title="Soft operating preference",
    ylabel="T [K]"))
display_plot(plot_series(soft_time, {"slack": soft_slack}, title="Paid operating-limit violation",
    ylabel="temperature slack [K]"))


### Explain the seven results

The energy-balance signs encode exothermic release and jacket exchange. The
linear model agrees locally but loses accuracy as the operating point moves;
RK4 refinement tests numerical integration separately. A fitted $U_A$ improves
prediction on held-out data, while the objective profile measures information
under stated assumptions. Temperature reveals concentration through reaction
heat, so a causal MHE can reduce a wrong concentration prior without observing
truth. Estimated-state feedback approaches ideal feedback while retaining noise
and startup errors. Positive slack honestly records an unattainable operating
preference. Finally, iteration exhaustion and a hard node-zero contradiction
require different remedies; neither justifies silently using failed values.

Try larger sensor noise, a shorter estimation window or a fitted-parameter offset
through the ordinary functions above. Explain changes in residuals, estimation
error, tracking and solve status together.
